### Langkah 6: Ekspor Model & End-to-End Pipeline ke File .pkl

Ini adalah langkah pamungkas dari proyek Machine Learning kita:
1. Membangun Scikit-Learn Pipeline Utuh:
   - Menggabungkan Preprocessor (Encoding + Scaling Rupiah) dan Model Balanced Random Forest ke dalam satu kesatuan sistem.
2. Menyimpan ke File .pkl (model_credit_risk_idr.pkl):
   - File otak ini bersifat mandiri (*self-contained*). Nanti bisa dipanggil di mana saja (Streamlit web app, backend Flask/FastAPI, atau aplikasi mobile).
3. Verifikasi & Uji Simulasi Nasabah Baru:
   - Menguji langsung file .pkl dengan data mentah nasabah baru tanpa perlu preprocessing manual lagi.

### 1. Import Library & Load Data Bersih IDR

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Deteksi lokasi file german_credit_clean_idr.csv
possible_paths = [
    "german_credit_clean_idr.csv",
    os.path.join("data", "german_credit_clean_idr.csv"),
    os.path.join("..", "data", "german_credit_clean_idr.csv")
]
DATA_PATH = next((p for p in possible_paths if os.path.exists(p)), None)
df = pd.read_csv(DATA_PATH)
X = df.drop(columns=["risk", "credit_amount"])
y = df["risk"]
print(f"✓ Data berhasil dimuat: {DATA_PATH}")
print(f"✓ Jumlah sampel: {len(df)} nasabah")

✓ Data berhasil dimuat: data\german_credit_clean_idr.csv
✓ Jumlah sampel: 1000 nasabah


### 2. Membangun Pipeline Lengkap (Preprocessor + Balanced Random Forest)
Pipeline ini membungkus:
- Transformasi fitur numerik Rupiah (StandardScaler)
- Transformasi kategori teks (OneHotEncoder)
- Algoritma klasifikasi Balanced Random Forest

In [2]:
numeric_features = [
    "duration_month",
    "credit_amount_idr",  # Nominal Pinjaman Rupiah
    "installment_rate",
    "residence_since",
    "age",
    "existing_credits",
    "num_dependents"
]
categorical_features = [col for col in X.columns if col not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), categorical_features)
    ]
)

full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42))
])

full_pipeline.fit(X, y)
print("✓ Full Pipeline berhasil dilatih pada seluruh data nasabah!")

✓ Full Pipeline berhasil dilatih pada seluruh data nasabah!


### 3. Ekspor & Simpan ke File .pkl

In [3]:
MODEL_FILE = "model_credit_risk_idr.pkl"
joblib.dump(full_pipeline, MODEL_FILE)
file_size_mb = os.path.getsize(MODEL_FILE) / (1024 * 1024)
print(f"✓ Model Pipeline berhasil disimpan ke: {MODEL_FILE}")
print(f"✓ Ukuran file model: {file_size_mb:.2f} MB")

✓ Model Pipeline berhasil disimpan ke: model_credit_risk_idr.pkl
✓ Ukuran file model: 3.92 MB


### 4. Uji Simulasi Nasabah Baru Menggunakan File .pkl
Sekarang kita simulasikan bagaimana aplikasi web atau backend bank menggunakan file .pkl ini secara praktis:

In [4]:
# 1. Muat kembali file .pkl yang sudah disimpan
loaded_pipeline = joblib.load("model_credit_risk_idr.pkl")
print("✓ File .pkl berhasil dimuat kembali!\n")

# 2. Definisikan 2 profil nasabah baru (Data mentah Rupiah)
nasabah_a = pd.DataFrame([{
    "checking_status": "A14",
    "duration_month": 12,
    "credit_history": "A32",
    "purpose": "A43",
    "savings_status": "A65",
    "employment_length": "A75",
    "installment_rate": 2,
    "personal_status_sex": "A93",
    "other_debtors": "A101",
    "residence_since": 4,
    "property": "A121",
    "age": 38,
    "other_installment_plans": "A143",
    "housing": "A152",
    "existing_credits": 1,
    "job": "A173",
    "num_dependents": 1,
    "telephone": "A192",
    "foreign_worker": "A201",
    "credit_amount_idr": 20000000  # Pinjaman Rp 20 Juta
}])

nasabah_b = pd.DataFrame([{
    "checking_status": "A11",
    "duration_month": 48,
    "credit_history": "A30",
    "purpose": "A40",
    "savings_status": "A61",
    "employment_length": "A71",
    "installment_rate": 4,
    "personal_status_sex": "A92",
    "other_debtors": "A101",
    "residence_since": 1,
    "property": "A124",
    "age": 20,
    "other_installment_plans": "A141",
    "housing": "A151",
    "existing_credits": 2,
    "job": "A171",
    "num_dependents": 2,
    "telephone": "A191",
    "foreign_worker": "A201",
    "credit_amount_idr": 120000000 # Pinjaman Rp 120 Juta
}])

# 3. Prediksi skor probabilitas langsung via pipeline .pkl
prob_a = loaded_pipeline.predict_proba(nasabah_a)[0, 1]
prob_b = loaded_pipeline.predict_proba(nasabah_b)[0, 1]

test_df = pd.DataFrame({
    "Profil Calon Nasabah": [
        "Nasabah A (Gaji mapan, pinjaman Rp 20jt, rumah sendiri)",
        "Nasabah B (Pengangguran, pinjaman Rp 120jt, saldo minus)"
    ],
    "Skor Kelayakan Probabilitas (%)": [
        f"{prob_a * 100:.1f}%",
        f"{prob_b * 100:.1f}%"
    ],
    "Status Keputusan Bank": [
        "DISETUJUI OTOMATIS (Risiko Sangat Rendah)" if prob_a >= 0.7 else "REVIEW",
        "DITOLAK OTOMATIS (Risiko Gagal Bayar Tinggi)" if prob_b < 0.5 else "REVIEW"
    ]
})
print("=== HASIL SIMULASI PREDIKSI NASABAH BARU ===")
test_df

✓ File .pkl berhasil dimuat kembali!

=== HASIL SIMULASI PREDIKSI NASABAH BARU ===


,Profil Calon Nasabah,Skor Kelayakan Probabilitas (%),Status Keputusan Bank
0,"Nasabah A (Gaji mapan, pinjaman Rp 20jt, rumah...",95.0%,DISETUJUI OTOMATIS (Risiko Sangat Rendah)
1,"Nasabah B (Pengangguran, pinjaman Rp 120jt, sa...",29.0%,DITOLAK OTOMATIS (Risiko Gagal Bayar Tinggi)


### Kesimpulan & Hasil Akhir Proyek
1. File model model_credit_risk_idr.pkl telah selesai dibuat dan tersimpan di folder proyek.
2. File .pkl ini sudah mencakup seluruh proses: menerima data mentah nasabah (dengan nilai nominal Rupiah), melakukan scaling & encoding otomatis, lalu mengeluarkan skor probabilitas kelayakan (0 - 100%).
3. Model ini siap dideploy ke aplikasi web (misal Streamlit) untuk demo interaktif!